**TE Differential Expression for PRC2 Loss Cohort**

This notebook repeats your TE differential expression analysis using the GENCODE v46-corrected PRC2 loss cohort for: EZH2 / EED / SUZ12 / RBBP4 / RBBP7 loss

Primary model: ~ matched_set + alteration_status

Sensitivity model: ~ alteration_status + tumour_type + sex


In [ ]:
# Load libraries and set paths

library(dplyr)
library(readr)
library(tibble)
library(Biobase)
library(limma)
library(stringr)
library(purrr)
library(ggplot2)

# Create and enter a fresh differential-expression directory for the
# GENCODE-corrected cohort.
project_root <- "/home/kennes38/ResearchProject"
analysis_dir <- file.path(project_root, "01_prc2_TE_differential_expression")
dir.create(analysis_dir, showWarnings = FALSE, recursive = TRUE)
setwd(analysis_dir)

gene_set_label <- "prc2_loss"

# REdiscoverTE expression data folder
tedata_root <- file.path(project_root, "TEdata_resources")

match_dir <- file.path(project_root, "01_prc2_WT_matching")

# All differential-expression outputs go into the current directory
te_de_output_dir <- "."

case_control_file <- file.path(
  match_dir,
  paste0(gene_set_label, "_case_control_matched_table_cancer_sex_primary_tumour_only.csv")
)

file.exists(case_control_file)
dir.exists(tedata_root)
dir.exists(te_de_output_dir)

In [ ]:
# Load REdiscoverTE ExpressionSet

rds_hits <- list.files(
  tedata_root,
  pattern = "eset_TCGA_TE_intergenic_logCPM\\.RDS$",
  recursive = TRUE,
  full.names = TRUE
)

stopifnot(length(rds_hits) == 1)
dat <- readRDS(rds_hits[1])

expr_full <- exprs(dat)

# Rows - features/subfamilies, columns - TCGA samples
# Values - REdiscoverTE-normalised log2CPM expression values
dim(expr_full)
summary(as.numeric(expr_full))

In [ ]:
# Load Matched case-control table

case_control_tbl <- read_csv(case_control_file, show_col_types = FALSE)

# Table has one row per case-control pairing
# Case with two matched WT controls appears twice
case_control_tbl %>%
  summarise(
    n_cases = n_distinct(case_sample_id_16),
    n_controls = n_distinct(WT_sample_id_16),
    n_rows = n()
  )

# QC - cases and controls should both be primary tumour code 01
case_control_tbl %>%
  count(case_sample_type_code, WT_sample_type_code)

case_control_tbl %>%
  distinct(case_sample_id_16, case_project) %>%
  count(case_project, sort = TRUE)

In [ ]:
# Build analysis metadata

case_meta <- case_control_tbl %>%
  # One metadata row for each altered case
  transmute(
    TEdata_full_id = case_TEdata_full_id,
    sample_id_16 = case_sample_id_16,
    patient_id = case_patient_id,
    sample_type_code = case_sample_type_code,
    matched_set = case_sample_id_16,

    # 'altered' - sample has LoF and/or HomDel in at least one gene from PRC2 gene set
    alteration_status = "altered",
    tumour_type = case_project,
    sex = case_sex,
    genes_lost,
    events,
    lof_genes,
    homdel_genes,
    mutation_classes,
    n_genes_lost,
    n_loss_events,
    has_lof,
    has_homdel
  ) %>%
  distinct()

control_meta <- case_control_tbl %>%
  # One metadata row for each WT control
  # Controls inherit the case sample ID as matched_set -> tells limma which controls belong to which altered case.
  transmute(
    TEdata_full_id = WT_TEdata_full_id,
    sample_id_16 = WT_sample_id_16,
    patient_id = WT_patient_id,
    sample_type_code = WT_sample_type_code,
    matched_set = case_sample_id_16,
    alteration_status = "WT",
    tumour_type = WT_project,
    sex = WT_sex,
    genes_lost = NA_character_,
    events = NA_character_,
    lof_genes = NA_character_,
    homdel_genes = NA_character_,
    mutation_classes = NA_character_,
    n_genes_lost = NA_integer_,
    n_loss_events = NA_integer_,
    has_lof = FALSE,
    has_homdel = FALSE
  ) %>%
  distinct()

analysis_meta <- bind_rows(case_meta, control_meta) %>%
  mutate(
    # WT is the reference level -> limma coefficient alteration_statusaltered means altered minus WT.
    alteration_status = factor(alteration_status, levels = c("WT", "altered")),
    matched_set = factor(matched_set),
    tumour_type = factor(tumour_type),
    sex = factor(sex)
  ) %>%
  distinct(TEdata_full_id, .keep_all = TRUE) %>%
  arrange(matched_set, alteration_status, TEdata_full_id)

analysis_meta %>% count(alteration_status)
analysis_meta %>% count(matched_set) %>% count(n, name = "n_matched_sets")

write_csv(
  analysis_meta,
  file.path(te_de_output_dir, paste0(gene_set_label, "_TE_analysis_sample_metadata.csv"))
)

In [ ]:
# Align expression matrix to metadata

missing_samples <- setdiff(analysis_meta$TEdata_full_id, colnames(expr_full))
missing_samples
stopifnot(length(missing_samples) == 0)

# Subset the full expression matrix to exactly the samples in the analysis
expr <- expr_full[, analysis_meta$TEdata_full_id, drop = FALSE]

# Check - limma assumes expression columns are in the same order as the rows of the metadata/design matrix.
stopifnot(identical(colnames(expr), analysis_meta$TEdata_full_id))

dim(expr)

In [ ]:
# Load TE feature annotation

annotation_hits <- list.files(
  tedata_root,
  pattern = "repName_repClass_repFam_TE\\.RDS$",
  recursive = TRUE,
  full.names = TRUE
)

annotation_hits
stopifnot(length(annotation_hits) == 1)

annotation_raw <- readRDS(annotation_hits[1])

# REdiscoverTE annotation may store the TE feature ID as an explicit feature_id column or as row names
annotation <- annotation_raw %>%
  as.data.frame() %>%
  tibble::rownames_to_column("feature_id_from_rownames") %>%
  as_tibble()

colnames(annotation)

if (!"feature_id" %in% colnames(annotation)) {
  if (any(annotation$feature_id_from_rownames %in% rownames(expr_full))) {
    annotation <- annotation %>%
      mutate(feature_id = feature_id_from_rownames)
  } else if ("repName" %in% colnames(annotation) && any(annotation$repName %in% rownames(expr_full))) {
    annotation <- annotation %>%
      mutate(feature_id = repName)
  } else {
    stop(
      "Could not identify feature_id in the TE annotation. ",
      "Check colnames(annotation) and rownames(expr_full)."
    )
  }
}

annotation <- annotation %>%
  rename(
    TE_subfamily = repName,
    TE_class = repClass,
    TE_family = repFam
  ) %>%
  select(feature_id, TE_subfamily, TE_class, TE_family, everything())

# Check annotation feature IDs match the expression matrix row names
cat("Annotated features overlapping expression matrix:", sum(annotation$feature_id %in% rownames(expr_full)), "\n")
cat("Expression features:", nrow(expr_full), "\n")

head(annotation)

In [ ]:
# Set limma helper function

run_limma <- function(expression_matrix, metadata, design_formula, coef_name, feature_level) {
  # Design matrix converts metadata columns into model covariates
  design <- model.matrix(design_formula, data = metadata)

  # lmFit fits linear model for every TE row
  fit <- lmFit(expression_matrix, design)

  # eBayes stabilises variance estimates across all tested TE features
  # trend = TRUE is appropriate for logCPM-style expression data where variance can depend on average expression
  fit <- eBayes(fit, trend = TRUE)

  # topTable extracts the coefficient of interest and applies BH FDR correction
  topTable(
    fit,
    coef = coef_name,
    number = Inf,
    adjust.method = "BH",
    sort.by = "P"
  ) %>%
    rownames_to_column("feature_id") %>%
    as_tibble() %>%
    mutate(feature_level = feature_level)
}

In [ ]:
# Run matched set model at feature/sub-family level

matched_feature_results <- run_limma(
  expression_matrix = expr,
  metadata = analysis_meta,
  design_formula = ~ matched_set + alteration_status,
  coef_name = "alteration_statusaltered",
  feature_level = "feature"
)

matched_subfamily_results <- matched_feature_results %>%
  # Add TE class/family/subfamily annotation for interpretation
  left_join(annotation, by = "feature_id") %>%
  mutate(feature_level = "subfamily")

head(matched_subfamily_results)

In [ ]:
# Aggregate expression to TE family level

expr_tbl <- as_tibble(expr, rownames = "feature_id", .name_repair = "minimal") %>%
  left_join(annotation, by = "feature_id")

family_expr_tbl <- expr_tbl %>%
  filter(!is.na(TE_family), TE_family != "") %>%
  select(-feature_id, -TE_subfamily, -TE_class) %>%
  group_by(TE_family) %>%
  summarise(across(where(is.numeric), \(x) mean(x, na.rm = TRUE)), .groups = "drop")

family_expr <- family_expr_tbl %>%
  column_to_rownames("TE_family") %>%
  as.matrix()

stopifnot(identical(colnames(family_expr), analysis_meta$TEdata_full_id))

family_results <- run_limma(
  expression_matrix = family_expr,
  metadata = analysis_meta,
  design_formula = ~ matched_set + alteration_status,
  coef_name = "alteration_statusaltered",
  feature_level = "family"
)

head(family_results)

In [ ]:
# Run covariate-sensitivity model

covariate_meta <- analysis_meta %>%
  filter(
    !is.na(tumour_type),
    !is.na(sex),
    tumour_type != "",
    sex != ""
  ) %>%
  droplevels()

expr_covariate <- expr[, covariate_meta$TEdata_full_id, drop = FALSE]

stopifnot(identical(colnames(expr_covariate), covariate_meta$TEdata_full_id))

covariate_subfamily_results <- run_limma(
  expression_matrix = expr_covariate,
  metadata = covariate_meta,
  design_formula = ~ alteration_status + tumour_type + sex,
  coef_name = "alteration_statusaltered",
  feature_level = "subfamily_covariate_model"
) %>%
  left_join(annotation, by = "feature_id")

# Rebuild family-level expression for the same complete-case samples
family_expr_covariate <- family_expr[, covariate_meta$TEdata_full_id, drop = FALSE]

stopifnot(identical(colnames(family_expr_covariate), covariate_meta$TEdata_full_id))

covariate_family_results <- run_limma(
  expression_matrix = family_expr_covariate,
  metadata = covariate_meta,
  design_formula = ~ alteration_status + tumour_type + sex,
  coef_name = "alteration_statusaltered",
  feature_level = "family_covariate_model"
)

cat("Samples in matched model:", nrow(analysis_meta), "\n")
cat("Samples in covariate complete-case model:", nrow(covariate_meta), "\n")

head(covariate_subfamily_results)

In [ ]:
# ERV/LTR-only matched model


# Restrict the analysis to LTR/ERV-like TE features before running limma.
# This reduces the testing burden and focuses on the biologically most relevant
# repeat classes for an ERV derepression project.
erv_ltr_features <- annotation %>%
  filter(
    TE_class == "LTR" |
      str_detect(TE_family, regex("ERV|LTR", ignore_case = TRUE)) |
      str_detect(TE_subfamily, regex("ERV|LTR", ignore_case = TRUE))
  ) %>%
  pull(feature_id) %>%
  intersect(rownames(expr))

length(erv_ltr_features)

expr_erv_ltr <- expr[erv_ltr_features, , drop = FALSE]

erv_ltr_results <- run_limma(
  expression_matrix = expr_erv_ltr,
  metadata = analysis_meta,
  design_formula = ~ matched_set + alteration_status,
  coef_name = "alteration_statusaltered",
  feature_level = "ERV_LTR_subfamily"
) %>%
  left_join(annotation, by = "feature_id")

head(erv_ltr_results)

In [ ]:
# Save main results

all_results <- bind_rows(
  matched_subfamily_results,
  family_results,
  covariate_subfamily_results,
  covariate_family_results,
  erv_ltr_results
)

significance_summary <- all_results %>%
  # Summarise how many tests pass nominal P-value and FDR thresholds.
  # Nominal P < 0.05 is exploratory; FDR thresholds are the stronger evidence.
  group_by(feature_level) %>%
  summarise(
    n_tested = n(),
    n_fdr_0_05 = sum(adj.P.Val < 0.05, na.rm = TRUE),
    n_fdr_0_10 = sum(adj.P.Val < 0.10, na.rm = TRUE),
    n_nominal_0_05 = sum(P.Value < 0.05, na.rm = TRUE),
    .groups = "drop"
  )

significance_summary

write_csv(matched_subfamily_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_subfamily_matched_limma_results.csv")))
write_csv(family_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_family_matched_limma_results.csv")))
write_csv(covariate_subfamily_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_subfamily_covariate_limma_results.csv")))
write_csv(covariate_family_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_family_covariate_limma_results.csv")))
write_csv(erv_ltr_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_ERV_LTR_matched_limma_results.csv")))
write_csv(all_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_limma_results_ALL_LEVELS.csv")))
write_csv(significance_summary, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_DE_significance_summary.csv")))

In [ ]:
# Plot Candidate Nominal Hits by Group and Tumour Type

# Choose nominal hits from the matched subfamily model.
candidate_hits <- matched_subfamily_results %>%
  # These are exploratory hits only unless adj.P.Val also passes an FDR cutoff.
  filter(P.Value < 0.05) %>%
  arrange(P.Value) %>%
  slice_head(n = 10)

candidate_hits %>%
  select(feature_id, TE_subfamily, TE_class, TE_family, logFC, P.Value, adj.P.Val)

plot_candidate <- function(candidate) {
  safe_candidate <- str_replace_all(candidate, "[^A-Za-z0-9_.-]", "_")

  plot_df <- tibble(
    TEdata_full_id = colnames(expr),
    expression = as.numeric(expr[candidate, ])
  ) %>%
    left_join(analysis_meta, by = "TEdata_full_id")

  p1 <- ggplot(plot_df, aes(x = alteration_status, y = expression, fill = alteration_status)) +
    geom_boxplot(outlier.shape = NA, alpha = 0.7) +
    geom_jitter(width = 0.15, alpha = 0.6, size = 1.5) +
    labs(
      title = candidate,
      x = NULL,
      y = "REdiscoverTE log2CPM"
    ) +
    theme_bw() +
    theme(legend.position = "none")

  p2 <- ggplot(plot_df, aes(x = interaction(tumour_type, alteration_status), y = expression, fill = alteration_status)) +
    geom_boxplot(outlier.shape = NA, alpha = 0.7) +
    geom_jitter(width = 0.15, alpha = 0.6, size = 1.2) +
    labs(
      title = paste(candidate, "by tumour type"),
      x = NULL,
      y = "REdiscoverTE log2CPM"
    ) +
    theme_bw() +
    theme(axis.text.x = element_text(angle = 90, hjust = 1))

  ggsave(file.path(te_de_output_dir, paste0(safe_candidate, "_boxplot_status.png")), p1, width = 5, height = 4, dpi = 300)
  ggsave(file.path(te_de_output_dir, paste0(safe_candidate, "_boxplot_tumour_type_status.png")), p2, width = 9, height = 5, dpi = 300)

  list(status_plot = p1, tumour_type_plot = p2)
}

if (nrow(candidate_hits) > 0) {
  plot_candidate(candidate_hits$feature_id[1])
}

In [ ]:
# Check direction consistency across tumour types

direction_by_tumour_type <- function(candidate) {
  # For each tumour type, calculate the raw mean expression difference:
  # altered mean log2CPM minus WT mean log2CPM.
  # This helps identify whether an overall signal is broadly consistent or
  # driven by one tumour type/outlier group.
  purrr::map_dfr(split(analysis_meta, analysis_meta$tumour_type), function(meta_t) {
    samples <- meta_t$TEdata_full_id
    values <- expr[candidate, samples]

    mean_loss <- mean(values[meta_t$alteration_status == "altered"], na.rm = TRUE)
    mean_wt <- mean(values[meta_t$alteration_status == "WT"], na.rm = TRUE)

    tibble(
      feature_id = candidate,
      tumour_type = as.character(unique(meta_t$tumour_type)),
      n_loss = sum(meta_t$alteration_status == "altered"),
      n_wt = sum(meta_t$alteration_status == "WT"),
      mean_loss = mean_loss,
      mean_wt = mean_wt,
      delta_log2CPM = mean_loss - mean_wt
    )
  })
}

direction_results <- purrr::map_dfr(candidate_hits$feature_id, direction_by_tumour_type)

write_csv(
  direction_results,
  file.path(te_de_output_dir, paste0(gene_set_label, "_candidate_direction_by_tumour_type.csv"))
)

head(direction_results)

In [ ]:
# Volcano plot for matched subfamily model

volcano_df <- matched_subfamily_results %>%
  # Volcano plots show effect size on the x-axis and statistical evidence
  # on the y-axis. Points far left/right and high up are the strongest signals.
  mutate(
    neg_log10_p = -log10(P.Value),
    result = case_when(
      adj.P.Val < 0.05 ~ "FDR < 0.05",
      P.Value < 0.05 ~ "Nominal P < 0.05",
      TRUE ~ "Not significant"
    )
  )

p_volcano <- ggplot(volcano_df, aes(x = logFC, y = neg_log10_p, colour = result)) +
  geom_point(alpha = 0.7, size = 1.8) +
  geom_vline(xintercept = 0, linetype = "dotted", colour = "grey60") +
  geom_hline(yintercept = -log10(0.05), linetype = "dashed", colour = "grey40") +
  scale_colour_manual(
    values = c(
      "FDR < 0.05" = "#D55E00",
      "Nominal P < 0.05" = "#2C6DB2",
      "Not significant" = "grey75"
    )
  ) +
  labs(
    title = "TE subfamily differential expression",
    subtitle = "PRC2 loss primary tumours vs matched WT primary tumour controls",
    x = "logFC",
    y = "-log10(P value)",
    colour = "Result"
  ) +
  theme_classic()

p_volcano

ggsave(
  file.path(te_de_output_dir, paste0(gene_set_label, "_TE_subfamily_volcano.png")),
  p_volcano,
  width = 7,
  height = 5,
  dpi = 300
)